# Reading SimpleFold's sampling trajectory

SimpleFold integrates **500 steps** per structure and discards the path. This logs per-residue
coordinates every 5th step and turns them into features describing *how* the model reached its
answer: convergence point, path length, tortuosity, late-stage jitter, and when the 6 samples
diverge. **No other model here has a trajectory** -- IgFold and NanoBodyBuilder2 are feed-forward.

Targets are a **random, unstratified** sample of 48. The previous run stratified by known error,
which inflated the spread-vs-error correlation (within-stratum rho fell to +0.09). This is the clean test.

In [ ]:
import os, sys, json, time, glob, subprocess, shutil
import numpy as np
PICKS = ['6X07B', '6Z6VG', '7D2ZA', '7JVBD', '7M1HE', '7NX0E', '7P5VG', '7Q3QB', '7QIAB', '7R4IE', '7R63C', '7SALD', '7UIAB', '7VPYA', '7XKAB', '7YZ9B', '7ZAUD', '8AOMV', '8B8IA', '8BF4B', '8EE2H', '8FTTA', '8HXRA', '8OPRB', '8PE1C', '8Q93C', '8QZ7D', '8RTFJ', '8S4UB', '8UW9B', '8V62E', '8WCGb', '8YBOB', '9E76Q', '9EN2B', '9FR4C', '9H6VE', '9HKQF', '9HZJA', '9HZKA', '9J7YH', '9JRSA', '9LM5D', '9NSDF', '9P6XH', '9QTOF', '9RN6B', '9RYFC']
NS, STEPS, TAU, SEED = '6', '500', '0.01', '12345'
EVERY = 5
WORK, SCRATCH = '/kaggle/working', '/tmp/sf'
TRAJ = f'{SCRATCH}/traj'; os.makedirs(TRAJ, exist_ok=True)
T0 = time.time()
R = {'n_samples': int(NS), 'steps': int(STEPS), 'tau': float(TAU), 'log_every': EVERY}
import torch
assert torch.cuda.device_count()==2 and torch.cuda.get_device_capability(0)>=(7,0)
R['gpus']=[torch.cuda.get_device_name(g) for g in range(2)]
_ds=glob.glob('/kaggle/input/**/targets_vhh.json',recursive=True); assert len(_ds)==1
TG={f"{t['pdb']}{t['H']}": t for t in json.load(open(_ds[0]))}
TG={k:TG[k] for k in PICKS}; assert len(TG)==48
print(R['gpus'], len(TG),'targets')
def must(c,w):
    p=subprocess.run(c,shell=True,text=True,capture_output=True)
    if p.returncode: raise RuntimeError(w+':\n'+(p.stdout+p.stderr)[-1500:])
must('git clone -q --depth 1 -b fix-multichain-fasta-parsing '
     'https://github.com/omar-A-hassan/ml-simplefold '+SCRATCH+'/repo','clone')
must(f'{sys.executable} -m pip install -q -e {SCRATCH}/repo','install')
must(f'{sys.executable} -m pip install -q git+https://github.com/facebookresearch/esm.git','esm')
must(f'{sys.executable} -m pip install -q biotite','biotite')

## Patch: fp16 ESM, name the trajectory, log the path

In [ ]:
INF = f'{SCRATCH}/repo/src/simplefold/inference.py'
s = open(INF).read()
a = '        esm_model = esm_model.to(device)'; assert s.count(a)==1
s = s.replace(a, '        esm_model = esm_model.half().to(device)')
a2 = '        sampled_coord, pad_mask, plddts = generate_structure('
assert s.count(a2)==1
s = s.replace(a2, "        os.environ['SF_TRAJ_NAME'] = str(record.id)\n" + a2)
assert 'import os' in s
open(INF,'w').write(s)

SMP = f'{SCRATCH}/repo/src/simplefold/model/torch/sampler.py'
t = open(SMP).read()
a3 = '        y_sampled = noise\n        feats = batch'
assert t.count(a3)==1, 'anchor a3'
hook_init = ('\n'
  "        _tdir = os.environ.get('SF_TRAJ_DIR'); _traj = []\n"
  "        _every = int(os.environ.get('SF_TRAJ_EVERY', '5'))\n"
  '        if _tdir:\n'
  "            _idx = feats['atom_to_token_idx']\n"
  "            _msk = feats['atom_pad_mask'].bool()\n"
  '            _L = int(_idx[_msk].max().item()) + 1')
t = t.replace(a3, a3 + hook_init)

a4 = ('            y_sampled = self.euler_maruyama_step(\n'
      '                model_fn,\n                flow,\n                y_sampled,\n'
      '                t,\n                t_next,\n                feats,\n            )')
assert t.count(a4)==1, 'anchor a4'
hook_log = ('\n'
  '            if _tdir and (i % _every == 0 or i == sampling_timesteps - 1):\n'
  '                _b = y_sampled.shape[0]\n'
  '                _acc = torch.zeros(_b, _L, 3, device=y_sampled.device, dtype=y_sampled.dtype)\n'
  '                _cnt = torch.zeros(_b, _L, 1, device=y_sampled.device, dtype=y_sampled.dtype)\n'
  '                for _j in range(_b):\n'
  '                    _m = _msk[_j] if _msk.shape[0] == _b else _msk[0]\n'
  '                    _ii = (_idx[_j] if _idx.shape[0] == _b else _idx[0])[_m]\n'
  '                    _acc[_j].index_add_(0, _ii, y_sampled[_j][_m])\n'
  '                    _cnt[_j].index_add_(0, _ii, torch.ones_like(y_sampled[_j][_m][:, :1]))\n'
  '                _traj.append((_acc / _cnt.clamp(min=1)).float().cpu().numpy())')
t = t.replace(a4, a4 + hook_log)

a5 = '        return {\n            \"denoised_coords\": y_sampled\n        }'
assert t.count(a5)==1, 'anchor a5'
save = ('        if _tdir and _traj:\n'
        '            import numpy as _np\n'
        "            _nm = os.environ.get('SF_TRAJ_NAME', 'unknown')\n"
        "            _np.save(os.path.join(_tdir, _nm + '.npy'), _np.stack(_traj))\n")
t = t.replace(a5, save + a5)
if not t.lstrip().startswith('import os'): t = 'import os\n' + t
open(SMP,'w').write(t)
print('patched: fp16 ESM + trajectory hook')

## Fold with trajectory logging

In [ ]:
fadir=f'{SCRATCH}/fa'; os.makedirs(fadir,exist_ok=True)
for tid,t in TG.items(): open(f'{fadir}/{tid}.fasta','w').write('>A|'+tid+'\n'+t['hseq']+'\n')
w=subprocess.run([sys.executable,'-c',
  "import torch,esm; torch.hub.load('facebookresearch/esm:main','esm2_t36_3B_UR50D')"],
  capture_output=True,text=True)
if w.returncode: raise RuntimeError((w.stdout+w.stderr)[-1500:])
tids=sorted(TG)
for g in (0,1):
    d=f'{SCRATCH}/sh{g}'; shutil.rmtree(d,ignore_errors=True); os.makedirs(d)
for i,tid in enumerate(tids): os.link(f'{fadir}/{tid}.fasta', f'{SCRATCH}/sh{i%2}/{tid}.fasta')
env0={**os.environ,'SF_TRAJ_DIR':TRAJ,'SF_TRAJ_EVERY':str(EVERY)}
t0=time.time(); procs=[]
for g in (0,1):
    if g: time.sleep(60)
    procs.append(subprocess.Popen(
        ['simplefold','--simplefold_model','simplefold_100M','--backend','torch',
         '--ckpt_dir',f'{SCRATCH}/artifacts','--num_steps',STEPS,'--tau',TAU,
         '--nsample_per_protein',NS,'--seed',SEED,
         '--fasta_path',f'{SCRATCH}/sh{g}','--output_dir',f'{SCRATCH}/out{g}'],
        stdout=subprocess.PIPE,stderr=subprocess.STDOUT,text=True,
        env={**env0,'CUDA_VISIBLE_DEVICES':str(g)}))
    print('launched',g,flush=True)
for g,p in enumerate(procs):
    o=p.communicate()[0]; R.setdefault('rc',{})[g]=p.returncode
    if p.returncode: print('GPU'+str(g)+' FAILED', o[-2000:])
R['fold_min']=round((time.time()-t0)/60,1)
R['n_cif']=len(glob.glob(f'{SCRATCH}/out*/**/*.cif',recursive=True))
R['n_traj']=len(glob.glob(f'{TRAJ}/*.npy'))
print(R['n_cif'],'cif,',R['n_traj'],'trajectories,',R['fold_min'],'min')
assert R['n_traj'] >= 0.9*len(TG), (R['n_traj'], len(TG))

## Trajectory features + structure scores

In [ ]:
import biotite.structure.io.pdbx as pdbx
def kabsch(P,Q):
    Pm,Qm=P.mean(0),Q.mean(0)
    V,_,W=np.linalg.svd((P-Pm).T@(Q-Qm)); D=np.diag([1,1,np.sign(np.linalg.det(V@W))])
    return V@D@W,Pm,Qm
rmsd=lambda a,b: float(np.sqrt(((a-b)**2).sum(-1).mean()))
AT=['N','CA','C','CB']; rows=[]
for tid,t in TG.items():
    f=f'{TRAJ}/{tid}.npy'
    g=sorted(glob.glob(f'{SCRATCH}/out*/**/*'+tid+'*.cif',recursive=True))
    if not os.path.exists(f) or len(g)<2: R.setdefault('skip',[]).append(tid); continue
    A=np.load(f); Ln=len(t['hseq'])
    if A.shape[2]!=Ln: R.setdefault('skip',[]).append(tid+':L'); continue
    lo,hi=t['cdr3']; c3=np.zeros(Ln,bool); c3[lo:hi]=True
    fwm=np.ones(Ln,bool)
    for k in ('cdr1','cdr2','cdr3'):
        a,b=t[k]; fwm[a:b]=False
    d=np.linalg.norm(np.diff(A,axis=0),axis=-1)
    def feat(m,pre):
        dm=d[:,:,m].mean(-1); tot=dm.sum(0)
        cum=np.cumsum(dm,0)/np.maximum(tot,1e-9)
        conv=(cum<0.9).sum(0)/dm.shape[0]
        net=np.linalg.norm(A[-1][:,m,:]-A[0][:,m,:],axis=-1).mean(-1)
        last=max(int(dm.shape[0]*0.2),1)
        return {pre+'_path':float(tot.mean()), pre+'_net':float(net.mean()),
                pre+'_tort':float((tot/np.maximum(net,1e-9)).mean()),
                pre+'_conv':float(conv.mean()), pre+'_jitter':float(dm[-last:].mean()),
                pre+'_early':float(dm[:last].mean())}
    row={'tid':tid,'cdr3_len':int(hi-lo),'L':Ln}
    row.update(feat(c3,'c3')); row.update(feat(fwm,'fw'))
    # UNALIGNED divergence (what the first run measured -- contaminated by rigid-body motion)
    div=np.array([float(np.nanmean(np.nanstd(A[s][:,c3,:],axis=0))) for s in range(A.shape[0])])
    row['div_final']=float(div[-1]); row['div_half']=float(div[len(div)//2])
    row['div_step90']=float((np.cumsum(div)/max(div.sum(),1e-9)<0.9).sum()/len(div))
    # ALIGNED divergence: superimpose the 6 samples on the framework AT EACH STEP first,
    # so what is left is genuine conformational disagreement, not global rotation.
    dva=[]
    for st in range(A.shape[0]):
        B=A[st]; b0=B[0]
        AA=[b0]
        for j in range(1,B.shape[0]):
            Rot,Pm,Qm=kabsch(B[j][fwm],b0[fwm]); AA.append((B[j]-Pm)@Rot+Qm)
        AA=np.stack(AA)
        dva.append(float(np.nanmean(np.nanstd(AA[:,c3,:],axis=0))))
    dva=np.array(dva)
    row['adiv_final']=float(dva[-1]); row['adiv_half']=float(dva[len(dva)//2])
    row['adiv_q1']=float(dva[len(dva)//4])
    row['adiv_slope']=float(np.polyfit(np.arange(len(dva)),dva,1)[0])
    row['adiv_step90']=float((np.cumsum(dva)/max(dva.sum(),1e-9)<0.9).sum()/len(dva))
    S=[]
    for fp in g:
        a=pdbx.get_structure(pdbx.CIFFile.read(fp),model=1)
        u=np.unique(a.res_id)
        if len(np.unique(a.chain_id))!=1 or len(u)!=Ln: break
        P=np.full((Ln,4,3),np.nan); rid=a.res_id-a.res_id.min()
        for k,an in enumerate(AT):
            mm=a.atom_name==an; P[rid[mm],k]=a.coord[mm]
        S.append(P)
    if len(S)<2: R.setdefault('skip',[]).append(tid+':parse'); continue
    ref=np.array([[[np.nan]*3 if v is None else v for v in r] for r in t['hca']],float)
    def err(P):
        ok=np.isfinite(ref[...,0])&np.isfinite(P[...,0]); sel=ok&fwm[:,None]
        if sel.sum()<60: return None
        Rot,Pm,Qm=kabsch(P[sel],ref[sel]); al=np.full_like(P,np.nan); al[ok]=(P[ok]-Pm)@Rot+Qm
        s2=ok&c3[:,None]
        return rmsd(al[s2],ref[s2]) if s2.sum()>=3 else None
    e=[err(P) for P in S]
    if any(x is None for x in e): R.setdefault('skip',[]).append(tid+':score'); continue
    base=S[0]; AL=[base]
    for P in S[1:]:
        m=np.isfinite(P[...,0])&np.isfinite(base[...,0])&fwm[:,None]
        Rot,Pm,Qm=kabsch(P[m],base[m]); AL.append((P-Pm)@Rot+Qm)
    AL=np.stack(AL)
    row['spread_cdr3']=float(np.nanmean(np.nanstd(AL,axis=0)[c3]))
    row['err']=round(float(e[0]),3); row['err_mean']=round(float(np.mean(e)),3)
    row['err_oracle']=round(float(min(e)),3)
    rows.append(row)
R['rows']=rows
KEEP=f'{WORK}/traj_npy'; os.makedirs(KEEP, exist_ok=True)
for f in glob.glob(f'{TRAJ}/*.npy'): shutil.copy(f, KEEP)
R['kept_mb']=round(sum(os.path.getsize(f) for f in glob.glob(f'{KEEP}/*.npy'))/1e6,1)
print('kept', len(os.listdir(KEEP)), 'trajectory arrays,', R['kept_mb'],'MB')
assert R['kept_mb'] < 2000
print(len(rows),'targets with trajectory+scores; skips',len(R.get('skip',[])))
json.dump(R, open(f'{WORK}/traj_results.json','w'), indent=1, default=str)
print('features:', [k for k in rows[0] if k!='tid'])